In [16]:
import polars as pl

Lê dados

In [17]:
df_participante = pl.scan_csv(
    source='pre_processamento/microdados_enem_2018/DADOS/MICRODADOS_ENEM_2018.csv', 
    infer_schema_length=10000, 
    has_header=True, 
    separator=';',
    encoding='utf8-lossy') 

In [18]:
df_participante.width, df_participante.select(pl.len()).collect().item()

C:\Users\greyc\AppData\Local\Temp\ipykernel_7436\470193877.py:1: PerformanceWarning: determining the width of a LazyFrame requires resolving its schema, which is a potentially expensive operation. Use `LazyFrame.collect_schema().len()` to get the width without this warning.
  df_participante.width, df_participante.select(pl.len()).collect().item()


(78, 5513733)

In [19]:
df_participante.schema

C:\Users\greyc\AppData\Local\Temp\ipykernel_7436\1103221186.py:1: PerformanceWarning: Resolving the schema of a LazyFrame is a potentially expensive operation. Use `LazyFrame.collect_schema()` to get the schema without this warning.
  df_participante.schema


Schema([('NU_INSCRICAO', Int64),
        ('NU_ANO', Int64),
        ('TP_FAIXA_ETARIA', Int64),
        ('TP_SEXO', String),
        ('TP_ESTADO_CIVIL', Int64),
        ('TP_COR_RACA', Int64),
        ('TP_NACIONALIDADE', Int64),
        ('TP_ST_CONCLUSAO', Int64),
        ('TP_ANO_CONCLUIU', Int64),
        ('TP_ESCOLA', Int64),
        ('TP_ENSINO', Int64),
        ('IN_TREINEIRO', Int64),
        ('CO_MUNICIPIO_ESC', Int64),
        ('NO_MUNICIPIO_ESC', String),
        ('CO_UF_ESC', Int64),
        ('SG_UF_ESC', String),
        ('TP_DEPENDENCIA_ADM_ESC', Int64),
        ('TP_LOCALIZACAO_ESC', Int64),
        ('TP_SIT_FUNC_ESC', Int64),
        ('CO_MUNICIPIO_PROVA', Int64),
        ('NO_MUNICIPIO_PROVA', String),
        ('CO_UF_PROVA', Int64),
        ('SG_UF_PROVA', String),
        ('TP_PRESENCA_CN', Int64),
        ('TP_PRESENCA_CH', Int64),
        ('TP_PRESENCA_LC', Int64),
        ('TP_PRESENCA_MT', Int64),
        ('CO_PROVA_CN', Int64),
        ('CO_PROVA_CH', Int64),
   

Seleciona colunas 

In [ ]:
df_participante = df_participante.drop([
    # Dados de Identificação 
	'NU_INSCRICAO', 'NU_ANO', 
    # Dados de indentificação da escola 
    'CO_MUNICIPIO_ESC', 'NO_MUNICIPIO_ESC', 'CO_UF_ESC', 'SG_UF_ESC',
	# Dados da localização da prova 
	'CO_MUNICIPIO_PROVA', 'NO_MUNICIPIO_PROVA', 'CO_UF_PROVA', 'SG_UF_PROVA',
	# Dados da prova 
    'TP_PRESENCA_CN', 'TP_PRESENCA_CH', 'TP_PRESENCA_LC', 'TP_PRESENCA_MT',
    'CO_PROVA_CN', 'CO_PROVA_CH', 'CO_PROVA_LC', 'CO_PROVA_MT', 
    'TP_STATUS_REDACAO', 
    # Vetor de respostas
    'TX_RESPOSTAS_CN', 'TX_RESPOSTAS_CH', 'TX_RESPOSTAS_LC', 'TX_RESPOSTAS_MT',
    'TX_GABARITO_CN', 'TX_GABARITO_CH', 'TX_GABARITO_LC', 'TX_GABARITO_MT',
    # Competencias redacao 
    'NU_NOTA_COMP1', 'NU_NOTA_COMP2', 'NU_NOTA_COMP3','NU_NOTA_COMP4', 'NU_NOTA_COMP5',
    ])

Limpa dados

In [21]:
df_participante.select(
    pl.all().null_count()
).collect()


TP_FAIXA_ETARIA,TP_SEXO,TP_ESTADO_CIVIL,TP_COR_RACA,TP_NACIONALIDADE,TP_ST_CONCLUSAO,TP_ANO_CONCLUIU,TP_ESCOLA,TP_ENSINO,IN_TREINEIRO,TP_DEPENDENCIA_ADM_ESC,TP_LOCALIZACAO_ESC,TP_SIT_FUNC_ESC,NU_NOTA_CN,NU_NOTA_CH,NU_NOTA_LC,NU_NOTA_MT,TP_LINGUA,NU_NOTA_REDACAO,Q001,Q002,Q003,Q004,Q005,Q006,Q007,Q008,Q009,Q010,Q011,Q012,Q013,Q014,Q015,Q016,Q017,Q018,Q019,Q020,Q021,Q022,Q023,Q024,Q025,Q026,Q027
u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32
0,0,217636,0,0,0,0,0,2030658,0,4064917,4064917,4064917,1608648,1365483,1365483,1608648,0,1365483,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,4,0


In [22]:
# substitui nulo por 'desconhecido'
df_participante = df_participante.with_columns(
    pl.col('TP_ESTADO_CIVIL').fill_null("DESCONHECIDO")
)

# remove as 4 linhas com Q026 nulo
df_participante = df_participante.drop_nulls(subset=["Q026"])

# supondo que se a nota estiver nula, quer dizer que a nota foi zero, substitui nulo por 0 
df_participante = df_participante.with_columns(
    pl.col('NU_NOTA_REDACAO').fill_null(0)
)


Removendo devido a alta quantidade de nulos



In [23]:
#df_participante = df_participante.drop(['TP_DEPENDENCIA_ADM_ESC', 'TP_LOCALIZACAO_ESC', 'TP_SIT_FUNC_ESC', 'TP_ENSINO'])

df_participante = df_participante.drop(['TP_SIT_FUNC_ESC', 'TP_ENSINO'])

df_participante = df_participante.drop_nulls(subset=["TP_LOCALIZACAO_ESC","TP_DEPENDENCIA_ADM_ESC" ])

Criando coluna Y

In [24]:
df_participante = df_participante.with_columns(
    (pl.sum_horizontal(pl.col("^NU_NOTA_.*$").fill_null(0)) / 5)
    .alias("nota_media")
)

In [25]:
df_participante = df_participante.drop([
   'NU_NOTA_CN', 'NU_NOTA_CH', 'NU_NOTA_LC', 'NU_NOTA_MT', 'NU_NOTA_REDACAO']) 

Converte para Parket

In [26]:
df_participante.width, df_participante.select(pl.len()).collect().item()

C:\Users\greyc\AppData\Local\Temp\ipykernel_7436\470193877.py:1: PerformanceWarning: determining the width of a LazyFrame requires resolving its schema, which is a potentially expensive operation. Use `LazyFrame.collect_schema().len()` to get the width without this warning.
  df_participante.width, df_participante.select(pl.len()).collect().item()


(40, 1448812)

In [27]:
df_participante.collect_schema()

Schema([('TP_FAIXA_ETARIA', Int64),
        ('TP_SEXO', String),
        ('TP_ESTADO_CIVIL', String),
        ('TP_COR_RACA', Int64),
        ('TP_NACIONALIDADE', Int64),
        ('TP_ST_CONCLUSAO', Int64),
        ('TP_ANO_CONCLUIU', Int64),
        ('TP_ESCOLA', Int64),
        ('IN_TREINEIRO', Int64),
        ('TP_DEPENDENCIA_ADM_ESC', Int64),
        ('TP_LOCALIZACAO_ESC', Int64),
        ('TP_LINGUA', Int64),
        ('Q001', String),
        ('Q002', String),
        ('Q003', String),
        ('Q004', String),
        ('Q005', Int64),
        ('Q006', String),
        ('Q007', String),
        ('Q008', String),
        ('Q009', String),
        ('Q010', String),
        ('Q011', String),
        ('Q012', String),
        ('Q013', String),
        ('Q014', String),
        ('Q015', String),
        ('Q016', String),
        ('Q017', String),
        ('Q018', String),
        ('Q019', String),
        ('Q020', String),
        ('Q021', String),
        ('Q022', String),
        ('Q

In [ ]:
df_participante.sink_parquet('alunos_enem_2018_processado.parquet')

: 